# 🔮 Future-Hour Transit Delay & Demand Forecasting
This notebook trains and evaluates multi-step time-series forecasting models predicting transit delays and demand for upcoming hours ($t+1, t+2, \dots, t+24$).

Key capabilities:
- Direct Multi-Horizon Gradient Boosting Regressors (LightGBM)
- Feature engineering with cyclic time encodings (sin/cos of hour and day-of-week)
- Lagged observations ($t-1, t-2, t-3, t-24$) and rolling window statistics
- Evaluation across forecast horizons (MAE, RMSE, R² degradation)

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))
from src.features import generate_synthetic_hourly_history, create_forecasting_features
from src.models.forecaster import FutureHourTransitForecaster

## 1. Prepare Hourly Dataset & Engineer Features

In [ ]:
# Generate 45 days of realistic Chicago hourly transit history
df_raw = generate_synthetic_hourly_history(num_days=45)
df_features = create_forecasting_features(df_raw)
print(f"Total feature matrix rows: {len(df_features)}, columns: {len(df_features.columns)}")
display(df_features.head())

## 2. Train Multi-Horizon Forecaster (t+1h to t+24h)

In [ ]:
forecaster = FutureHourTransitForecaster(target_col="delay_min")
horizons = [1, 2, 3, 6, 12, 24]
metrics = forecaster.train_multi_horizon_models(df_features, horizons=horizons)

metrics_df = pd.DataFrame(metrics).T
display(metrics_df)

## 3. Visualize Horizon Degradation Curve

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.plot(metrics_df['horizon_hours'], metrics_df['MAE'], marker='o', color='#e74c3c', linewidth=2, label='MAE (minutes)')
plt.plot(metrics_df['horizon_hours'], metrics_df['RMSE'], marker='s', color='#3498db', linewidth=2, label='RMSE (minutes)')
plt.title('Forecast Error Degradation Across Future Horizons (CTA Buses)')
plt.xlabel('Forecast Horizon (Hours Ahead)')
plt.ylabel('Prediction Error (Minutes)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

## 4. Save Model Artifacts

In [ ]:
saved_path = forecaster.save_model_artifacts()
print(f"Forecaster artifact saved to {saved_path}")